# Scryfall Card Tag Lookup

Paste a decklist and see the Scryfall Tagger oracle tags for every card in it, aggregated by frequency (weighted by copies).

Lines like `1 Hearthhull, the Worldseed (EOC) 1 *F*` are parsed automatically — quantity, set code, collector number, and foil marker are all optional; only the card name is required.

Note: tags aren't in the normal card API response — they only exist in Scryfall's `oracle-tags` bulk data file (gzipped JSONL), which this downloads once and keeps in memory.

In [8]:
import gzip, json, requests
from collections import Counter

HEADERS = {"User-Agent": "CardTagLookup/1.0", "Accept": "*/*"}

def bulk_jsonl(tag_type):
    meta = requests.get(f"https://api.scryfall.com/bulk-data/{tag_type}", headers=HEADERS).json()
    raw = requests.get(meta["jsonl_download_uri"], headers=HEADERS).content
    try:
        raw = gzip.decompress(raw)  # files are gzipped JSONL; skip if already decoded in transit
    except OSError:
        pass
    return [json.loads(line) for line in raw.decode("utf-8").splitlines() if line]

print("Downloading tag data from Scryfall (oracle tags)...")
oracle_tags = bulk_jsonl("oracle-tags")

# Build id -> tag label lookups (direct taggings only; ignores tag hierarchy)
oracle_map = {}
for tag in oracle_tags:
    for tg in tag["taggings"]:
        oracle_map.setdefault(tg["oracle_id"], []).append(tag["label"])

print(f"Loaded {len(oracle_map)} oracle-tagged cards.")


Loaded 35967 oracle-tagged cards.


In [9]:
import re

decklist_text = """
1 Aftermath Analyst
1 Ancient Greenwarden
1 Augur of Autumn
1 Azusa, Lost but Seeking
1 Bala Ged Recovery
1 Beast Within
1 Blasphemous Act
1 Blazemire Verge
1 Blood Crypt
1 Bloodstained Mire
1 Bojuka Bog
1 Braids, Arisen Nightmare
1 Cabaretti Courtyard
1 Cinder Glade
1 Command Tower
1 Conduit of Worlds
1 Constant Mists
1 Crop Rotation
1 Day of Black Sun
1 Enduring Vitality
1 Escape Tunnel
1 Eumidian Hatchery
1 Evendo Brushrazer
1 Exploration Broodship
1 Fabled Passage
1 Farseek
1 Field of the Dead
5 Forest
1 Harrow
1 Horizon Explorer
1 Icetill Explorer
1 Infernal Grasp
1 Insidious Fungus
1 Iridescent Vinelasher
1 Karplusan Forest
1 Korvold, Fae-Cursed King
1 Llanowar Wastes
1 Lotus Cobra
1 Lotus Field
1 Maestros Theater
1 Mayhem Devil
1 Mazirek, Kraul Death Priest
1 Mightform Harmonizer
2 Mountain
1 Mountain
1 Nahiri's Lithoforming
1 Nature's Lore
1 Noxious Revival
1 Oracle of Mul Daya
1 Overgrown Tomb
1 Overlord of the Hauntwoods
1 Putrefy
1 Rakdos Charm
1 Ramunap Excavator
1 Regrowth
1 Riveteers Overlook
1 Roiling Regrowth
1 Rootbound Crag
1 Sabotender
1 Scapeshift
1 Scouring Swarm
1 Scute Swarm
1 Six
1 Smoldering Marsh
1 Sol Ring
1 Soul of Windgrace
1 Splendid Reclamation
1 Sprouting Goblin
1 Stomping Ground
1 Sulfurous Springs
5 Swamp
1 Sylvan Library
1 Sylvan Safekeeper
1 Szarel, Genesis Shepherd
1 Tear Asunder
1 The Gitrog Monster
1 Thornspire Verge
1 Tifa Lockhart
1 Traveling Chocobo
1 Tunneling Geopede
1 Untimely Malfunction
1 Urza's Cave
1 Vampiric Tutor
1 Verdant Catacombs
1 Vernal Fen
1 Wastewood Verge
1 Wooded Foothills
1 Woodland Cemetery
1 Worldsoul's Rage
1 Zuran Orb
1 Hearthhull, the Worldseed
"""  # <-- replace with your own pasted decklist

def parse_decklist(text):
    entries = []
    for line in text.strip().splitlines():
        line = line.strip()
        m = re.match(r"^(?:(\d+)\s+)?(.+)$", line)
        if not m:
            continue
        qty = int(m.group(1)) if m.group(1) else 1
        name = re.sub(r"\s*\([A-Za-z0-9]{2,5}\)\s*\S*(\s*\*F\*)?\s*$", "", m.group(2)).strip()
        entries.append((qty, name))
    return entries

entries = parse_decklist(decklist_text)
quantities = {}
for qty, name in entries:
    quantities[name] = quantities.get(name, 0) + qty  # sum if a name appears on multiple lines
unique_names = list(dict.fromkeys(name for _, name in entries))

def extract_card_data(card_json):
    """Extract only mana cost, type, description (oracle text), and oracle_id from Scryfall card data."""
    name = card_json.get("name", "")
    oracle_id = card_json.get("oracle_id")
    
    # Handle multi-faced cards (e.g., MDFCs, Transform, Adventure, Split)
    if "card_faces" in card_json and card_json["card_faces"] and "mana_cost" in card_json["card_faces"][0]:
        faces = card_json["card_faces"]
        mana_cost = " // ".join(f.get("mana_cost", "") for f in faces)
        type_line = card_json.get("type_line") or " // ".join(f.get("type_line", "") for f in faces)
        description = "\n//\n".join(f.get("oracle_text", "") for f in faces)
        if not oracle_id and faces[0].get("oracle_id"):
            oracle_id = faces[0].get("oracle_id")
    else:
        mana_cost = card_json.get("mana_cost", "")
        type_line = card_json.get("type_line", "")
        description = card_json.get("oracle_text", "")
        
    return {
        "name": name,
        "oracle_id": oracle_id,
        "mana_cost": mana_cost,
        "type": type_line,
        "description": description
    }

def fetch_cards(names):
    cards = {}
    for i in range(0, len(names), 75):
        chunk = names[i:i + 75]
        body = {"identifiers": [{"name": n} for n in chunk]}
        data = requests.post("https://api.scryfall.com/cards/collection", json=body, headers=HEADERS).json()
        for c in data.get("data", []):
            extracted = extract_card_data(c)
            cards[extracted["name"]] = extracted
        for nf in data.get("not_found", []):
            print(f"  Not found: {nf.get('name')}")
    return cards

cards = fetch_cards(unique_names)

tag_counts = Counter()
for name in unique_names:
    card = cards.get(name)
    if not card:
        continue
    qty = quantities[name]
    tags = set(oracle_map.get(card.get("oracle_id"), []))
    cost_str = f" [{card['mana_cost']}]" if card.get("mana_cost") else ""
    type_str = f" — {card['type']}" if card.get("type") else ""
    print(f"  {card['name']} x{qty}{cost_str}{type_str}: {', '.join(sorted(tags)) or '(no tags found)'}")
    tag_counts.update({t: qty for t in tags})

print("\nTags across the decklist, most common first:")
for tag, n in tag_counts.most_common():
    print(f"  {n:>2}  {tag}")

def highlight_least_similar_cards(unique_names, cards, oracle_map, top_n=10):
    """
    Identifies and prints the top N cards with the least tag similarity
    (average Jaccard similarity) compared to all other cards in the deck.
    """
    card_tags = {}
    for name in unique_names:
        card = cards.get(name)
        if card:
            card_tags[name] = set(oracle_map.get(card.get("oracle_id"), []))

    if len(card_tags) <= 1:
        print("Not enough cards to compare similarity.")
        return []

    scores = []
    for name, tags in card_tags.items():
        other_similarities = []
        best_match_name = None
        best_match_sim = -1.0
        shared_with_best = set()

        for other_name, other_tags in card_tags.items():
            if other_name == name:
                continue
            # Jaccard similarity: |A ∩ B| / |A ∪ B|
            union = tags | other_tags
            sim = len(tags & other_tags) / len(union) if union else 0.0
            other_similarities.append(sim)
            if sim > best_match_sim:
                best_match_sim = sim
                best_match_name = other_name
                shared_with_best = tags & other_tags

        avg_sim = sum(other_similarities) / len(other_similarities) if other_similarities else 0.0
        scores.append({
            "name": name,
            "avg_similarity": avg_sim,
            "tag_count": len(tags),
            "tags": tags,
            "best_match": best_match_name,
            "best_match_sim": best_match_sim,
            "shared_with_best": shared_with_best
        })

    # Sort ascending by average similarity (lowest similarity first)
    scores.sort(key=lambda x: (x["avg_similarity"], x["tag_count"]))

    print(f"\n{'='*60}")
    print(f"Top {min(top_n, len(scores))} Cards with LEAST Tag Similarity (Outliers):")
    print(f"{'='*60}")
    for rank, item in enumerate(scores[:top_n], 1):
        card = cards.get(item["name"], {})
        cost_str = f" [{card.get('mana_cost')}]" if card.get("mana_cost") else ""
        type_str = f" — {card.get('type')}" if card.get("type") else ""
        tags_str = ", ".join(sorted(item["tags"])) if item["tags"] else "(no tags)"
        print(f"\n{rank:>2}. {item['name']} x{quantities.get(item['name'], 1)}{cost_str}{type_str}")
        print(f"    Avg Similarity: {item['avg_similarity']:.2%}")
        if card.get("description"):
            desc_lines = card["description"].splitlines()
            formatted_desc = "\n        ".join(desc_lines)
            print(f"    Description:\n        {formatted_desc}")
        print(f"    Tags ({item['tag_count']}): {tags_str}")
        if item["best_match"] and item["best_match_sim"] > 0:
            shared_str = ", ".join(sorted(item["shared_with_best"]))
            print(f"    Closest match: {item['best_match']} ({item['best_match_sim']:.1%} similarity; shared: {shared_str})")
        else:
            print(f"    Closest match: None (0% overlap with any card)")

    return scores[:top_n]

highlight_least_similar_cards(unique_names, cards, oracle_map, top_n=10)


  Aftermath Analyst x1 [{1}{G}] — Creature — Elf Detective: activated ability, alliteration, martyr, mass reanimation, mill-self, multi land ramp, reanimate-land, triggered ability
  Ancient Greenwarden x1 [{4}{G}{G}] — Creature — Elemental: crucible of worlds, cycle-znr-m-mono-creature, landfall, trigger doubler
  Augur of Autumn x1 [{1}{G}{G}] — Creature — Human Druid: alliteration, power matters, precognition engine
  Azusa, Lost but Seeking x1 [{2}{G}] — Legendary Creature — Human Monk: multi land ramp, play additional land
  Beast Within x1 [{2}{G}] — Instant: color break, donate token, removal-destroy, removal-permanent, single target instant/sorcery, spot removal, swap removal
  Blasphemous Act x1 [{8}{R}] — Sorcery: affinity for creatures, burn creature, hate-wide, sweeper, symmetrical
  Blazemire Verge x1 — Land: activated ability, cycle-dsk-verge, synergy-mountain, synergy-swamp
  Blood Crypt x1 — Land — Swamp Mountain: activated ability, cycle-rav-shockland, nonbasic-basic-l

[{'name': 'Blasphemous Act',
  'avg_similarity': 0.004273504273504273,
  'tag_count': 5,
  'tags': {'affinity for creatures',
   'burn creature',
   'hate-wide',
   'sweeper',
   'symmetrical'},
  'best_match': 'Rakdos Charm',
  'best_match_sim': 0.15384615384615385,
  'shared_with_best': {'hate-wide', 'symmetrical'}},
 {'name': 'Vampiric Tutor',
  'avg_similarity': 0.00505050505050505,
  'tag_count': 5,
  'tags': {'cycle-1mv-tutor',
   'cycle-ema-tutor',
   'life payment',
   'tutor-card',
   'tutor-to-top'},
  'best_match': 'Bloodstained Mire',
  'best_match_sim': 0.1111111111111111,
  'shared_with_best': {'life payment'}},
 {'name': 'Horizon Explorer',
  'avg_similarity': 0.006196790252517807,
  'tag_count': 8,
  'tags': {'attack trigger',
   'attacking matters',
   'combat ramp',
   'etb-untapper',
   'multi land ramp',
   'per-player',
   'repeatable landers',
   'untapper-land'},
  'best_match': 'Azusa, Lost but Seeking',
  'best_match_sim': 0.1111111111111111,
  'shared_with_bes